# Stage 2 — Demand Classification
### Walmart M5, store `CA_1`, department `FOODS_3` — full catalog

Loads the artifacts produced by `python -m src.analysis.demand_classification`
(which itself calls only reusable functions from `src/analysis/` — no
classification math is reimplemented here) and reports on them:
category counts, the classification table, the representative
selection, ADI/CV² distributions, and an aggregate-series preview.

No forecasting model, forecast, or forecast evaluation appears in this
notebook — Phase 1 Synopsis rules exclude that entirely. Run
`python -m src.data.preprocessor` and then
`python -m src.analysis.demand_classification` first if the files
below don't exist yet.

In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing config.yaml is
    found. Jupyter's kernel working directory isn't consistent across
    setups (often the notebook's own folder, sometimes the launch
    directory), so this avoids assuming which one it is.
    """
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate
    raise FileNotFoundError(
        "Could not find config.yaml in any parent directory of "
        f"'{start}'. Run this notebook from within the project."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.analysis.eda import plot_adi_cv2_quadrants, summarize_demand_classes
from src.data.validator import check_duplicate_keys, check_date_continuity
from src.utils.config import load_config

%matplotlib inline
pd.set_option("display.max_columns", None)

In [ ]:
config = load_config()
processed_dir = PROJECT_ROOT / config["data"]["processed_dir"]

classification_path = processed_dir / config["output"]["classification_filename"]
selected_path = processed_dir / config["output"]["selected_items_filename"]
aggregate_path = processed_dir / config["output"]["aggregate_filename"]

for p in [classification_path, selected_path, aggregate_path]:
    if not p.exists():
        raise FileNotFoundError(
            f"'{p}' not found. Run `python -m src.analysis.demand_classification` first."
        )

classification = pd.read_parquet(classification_path)
selected = pd.read_parquet(selected_path)
aggregate = pd.read_parquet(aggregate_path)

adi_threshold = config["classification"]["adi_threshold"]
cv2_threshold = config["classification"]["cv2_threshold"]

print(f"Loaded {len(classification)} classified items, "
      f"{len(selected)} selected, {len(aggregate)}-day aggregate series")

## 1. How many items, and how are they distributed

In [ ]:
n_total = len(classification)
print(f"Total CA_1 / FOODS_3 items analyzed: {n_total}")

summary = summarize_demand_classes(classification).set_index("demand_class")
summary

`NoDemand` (all-zero series — ADI and CV² undefined) and
`InsufficientData` (exactly one non-zero day — ADI defined, CV² isn't,
since a standard deviation needs at least two points) are not dropped
from the table above, and not an error — they're the honest result for
items that don't fit the four-quadrant scheme. They're excluded only
from the representative *selection* below, not from this table.

## 2. Classification table (sample)

In [ ]:
classification.sort_values(["demand_class", "item_id"]).head(15)

## 3. Representative selection

Up to 5 items per standard category (Smooth / Intermittent / Erratic /
Lumpy), sampled with a fixed seed for reproducibility. A category with
fewer than 5 items contributes all of it has — nothing here is padded
to look like a full set.

In [ ]:
selected_counts = selected["demand_class"].value_counts()
print("Selected counts by category:")
print(selected_counts)
print()
for category, group in selected.groupby("demand_class", observed=True):
    print(f"{category} ({len(group)}): {sorted(group['item_id'].tolist())}")

In [ ]:
selected.sort_values(["demand_class", "item_id"])

## 4. ADI distribution

In [ ]:
n_adi_undefined = classification["ADI"].isna().sum()
print(f"Items excluded from this histogram (ADI undefined -- all-zero series): {n_adi_undefined}")

fig, ax = plt.subplots(figsize=(7, 4))
classification["ADI"].dropna().plot(kind="hist", bins=15, ax=ax)
ax.axvline(adi_threshold, linestyle="--", color="gray", label=f"threshold = {adi_threshold}")
ax.set_xlabel("ADI (Average Demand Interval)")
ax.set_title("ADI distribution — CA_1 / FOODS_3")
ax.legend()
plt.tight_layout()
plt.show()

## 5. CV² distribution

In [ ]:
n_cv2_undefined = classification["CV2"].isna().sum()
print(f"Items excluded from this histogram (CV2 undefined -- 0 or 1 non-zero days): {n_cv2_undefined}")

fig, ax = plt.subplots(figsize=(7, 4))
classification["CV2"].dropna().plot(kind="hist", bins=15, ax=ax)
ax.axvline(cv2_threshold, linestyle="--", color="gray", label=f"threshold = {cv2_threshold}")
ax.set_xlabel("CV² (squared coefficient of variation, non-zero demand size)")
ax.set_title("CV² distribution — CA_1 / FOODS_3")
ax.legend()
plt.tight_layout()
plt.show()

## 6. ADI vs CV² — the four quadrants

Only items with both ADI and CV² defined can appear here (NoDemand and
InsufficientData items have no coordinates to plot); their counts are
reported above, not silently omitted.

In [ ]:
n_plotted = classification.dropna(subset=["ADI", "CV2"]).shape[0]
print(f"Plotting {n_plotted} of {n_total} items "
      f"({n_total - n_plotted} excluded: undefined ADI and/or CV2)")

fig, ax = plt.subplots(figsize=(8, 7))
plot_adi_cv2_quadrants(classification, adi_threshold, cv2_threshold, ax=ax)
ax.set_title("Demand classification quadrants — CA_1 / FOODS_3")
plt.tight_layout()
plt.show()

## 7. Aggregate series preview

Sum of the entire CA_1 / FOODS_3 catalog — every item, not just the
~20 selected above. This is what makes it dense enough to be useful
for aggregate-level modeling later; a purely descriptive view here,
not a forecast.

In [ ]:
print("Shape:", aggregate.shape)
print("Date range:", aggregate["date"].min().date(), "to", aggregate["date"].max().date())
aggregate.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(aggregate["date"], aggregate["sales"])
ax.set_xlabel("Date")
ax.set_ylabel("Total units sold")
ax.set_title("Aggregate daily demand, full CA_1 / FOODS_3 catalog")
plt.tight_layout()
plt.show()